Load the raw CSVs, sanity-check the data, reshape from 2 rows/match to 1 row/match,
and define the train / validation / holdout split.

Outputs written to this folder:
- `train_matches.parquet`   — 2012–2022, 1 row per match
- `holdout_matches.parquet` — 2023–2026, no corners column
- `split_config.json`       — which seasons belong to train vs validation

In [ ]:
import json
import os

import pandas as pd
from pathlib import Path

SOLUTION_DIR = Path.cwd()
DATA_DIR     = SOLUTION_DIR.parent

FEATURE_COLS = [f'feature_{i}' for i in range(1, 11)]

## Load

In [ ]:
train   = pd.read_csv(DATA_DIR / 'match_data.csv',                   parse_dates=['date'])
holdout = pd.read_csv(DATA_DIR / 'match_data_holdout_features.csv',  parse_dates=['date'])

print(f'train   : {len(train):,} rows   seasons {train.season.min()}–{train.season.max()}')
print(f'holdout : {len(holdout):,} rows  dates {holdout.date.min().date()} → {holdout.date.max().date()}')
print()
train.head(3)

## Target: corners distribution

Corners are count data (non-negative integers).
We want to know: roughly Poisson, or wider?  Do home teams get more?

In [ ]:
c = train['corners']
print(f'mean  : {c.mean():.2f}')
print(f'std   : {c.std():.2f}')
print(f'var/mean (overdispersion) : {c.var()/c.mean():.2f}  — Poisson = 1.0, NegBin > 1.0')

home_c = train.loc[train['home'] == 1, 'corners']
away_c = train.loc[train['home'] == 0, 'corners']
print(f'home avg: {home_c.mean():.3f}   away avg: {away_c.mean():.3f}')

## Features: range and correlation with corners

In [ ]:
print('Feature ranges:')
print(train[FEATURE_COLS].agg(['min', 'max', 'mean']).round(3).to_string())
print()
corr = train[FEATURE_COLS].corrwith(train['corners']).sort_values(key=abs, ascending=False)
print('Correlation with corners:')
print(corr.round(4))

## Reshape: 2 rows/match → 1 row/match

The Elo system needs to see both teams at once. Right now we have a home row and an away row per match — join them.

In [ ]:
def reshape_to_match_level(df: pd.DataFrame, has_corners: bool = True) -> pd.DataFrame:
    """Pivot from 2 rows per match to 1 row per match."""
    home = df[df['home'] == 1].copy()
    away = df[df['home'] == 0].copy()

    home_rename = {'team': 'home_team', 'opponent': 'away_team'}
    away_rename = {'team': 'away_team', 'opponent': 'home_team'}
    if has_corners:
        home_rename['corners'] = 'home_corners'
        away_rename['corners'] = 'away_corners'
    for col in FEATURE_COLS:
        home_rename[col] = f'home_{col}'
        away_rename[col] = f'away_{col}'

    home = home.rename(columns=home_rename)
    away = away.rename(columns=away_rename)

    home_keep = (['match_id', 'season', 'date', 'home_team', 'away_team']
                 + [f'home_{c}' for c in FEATURE_COLS]
                 + (['home_corners'] if has_corners else []))
    away_keep = (['match_id']
                 + [f'away_{c}' for c in FEATURE_COLS]
                 + (['away_corners'] if has_corners else []))

    merged = home[home_keep].merge(away[away_keep], on='match_id', how='inner')
    assert merged['match_id'].nunique() == len(merged)

    return merged.sort_values(['date', 'match_id']).reset_index(drop=True)


train_matches   = reshape_to_match_level(train,   has_corners=True)
holdout_matches = reshape_to_match_level(holdout, has_corners=False)

print(f'train_matches   : {train_matches.shape}')
print(f'holdout_matches : {holdout_matches.shape}')
train_matches.head(3)

## Train / validation split

- **Train** 2012–2019 — what the model learns from
- **Validation** 2020–2022 — compare against baseline here
- **Holdout** 2023–2026 — no labels, this is what we predict

In [ ]:
TRAIN_SEASONS = list(range(2012, 2020))
VAL_SEASONS   = [2020, 2021, 2022]

train_only = train_matches[train_matches['season'].isin(TRAIN_SEASONS)]
val        = train_matches[train_matches['season'].isin(VAL_SEASONS)]

print(f'Train : {len(train_only)} matches  ({TRAIN_SEASONS[0]}–{TRAIN_SEASONS[-1]})')
print(f'Val   : {len(val)} matches  ({VAL_SEASONS})')
print(f'Holdout: {len(holdout_matches)} matches  (2023–2026)')

## Save

In [ ]:
train_matches.to_parquet(  SOLUTION_DIR / 'train_matches.parquet',   index=False)
holdout_matches.to_parquet(SOLUTION_DIR / 'holdout_matches.parquet', index=False)

split_config = {'train_seasons': TRAIN_SEASONS, 'val_seasons': VAL_SEASONS}
with open(SOLUTION_DIR / 'split_config.json', 'w') as f:
    json.dump(split_config, f)

print('Saved train_matches.parquet, holdout_matches.parquet, split_config.json')